In [ ]:
#Bài 1
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
X = df[["thu_nhap", "diem_chi_tieu"]]
Xs = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
nhan = km.labels_

def silhouette_mot_diem(X, nhan, i):
    """Tính a, b và s cho điểm thứ i theo định nghĩa"""
    diem_i = X[i]
    cum_i = nhan[i]
    
    diem_cung_cum = X[nhan == cum_i]
    if len(diem_cung_cum) <= 1:
        a = 0.0
    else:
        kc_cung = np.sqrt(np.sum((diem_cung_cum - diem_i)**2, axis=1))
        a = np.sum(kc_cung) / (len(diem_cung_cum) - 1)
        
    cac_cum_khac = set(nhan) - {cum_i}
    b_min = float('inf')
    for c in cac_cum_khac:
        diem_khac = X[nhan == c]
        kc_khac = np.sqrt(np.sum((diem_khac - diem_i)**2, axis=1))
        b_c = np.mean(kc_khac)
        if b_c < b_min:
            b_min = b_c
    b = b_min
    
    mau = max(a, b)
    if mau == 0:
        return 0.0
    return (b - a) / mau

s_danh_sach = [silhouette_mot_diem(Xs, nhan, i) for i in range(len(Xs))]
print(f"Silhouette trung bình tự tính: {np.mean(s_danh_sach):.4f}")
print(f"Silhouette score từ thư viện:  {silhouette_score(Xs, nhan):.4f}")

min_idx = np.argmin(s_danh_sach)
print(f"Khách hàng có s nhỏ nhất (chỉ số {min_idx}): s = {s_danh_sach[min_idx]:.4f}")

Silhouette trung bình tự tính: 0.7227
Silhouette score từ thư viện:  0.7227
Khách hàng có s nhỏ nhất (chỉ số 113): s = 0.0286


Khách hàng có chỉ số $s$ nhỏ nhất (thậm chí có thể âm) là những điểm nằm nằm sát ranh giới giữa các cụm hoặc nằm lấp lửng giữa hai nhóm, nơi khoảng cách tới các điểm cùng cụm ($a$) lớn gần bằng hoặc lớn hơn khoảng cách tới cụm gần nhất khác ($b$).

In [3]:
#Bài 2
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
X3 = df[["tuoi", "thu_nhap", "diem_chi_tieu"]]
Xs3 = StandardScaler().fit_transform(X3)

print("K | Silhouette Score")
print("-" * 20)
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs3)
    score = silhouette_score(Xs3, km.labels_)
    print(f"{k:2d} | {score:.4f}")

K | Silhouette Score
--------------------
 2 | 0.3730
 3 | 0.4220
 4 | 0.4558
 5 | 0.4808
 6 | 0.4760
 7 | 0.4627
 8 | 0.4596
 9 | 0.4622
10 | 0.4573


Khi thêm cột tuoi vào, do phân bố độ tuổi của khách hàng có tính liên tục và không phân tách thành các cụm rõ rệt như không gian 2D (thu nhập và điểm chi tiêu), chỉ số silhouette tối ưu thường sẽ dịch chuyển (ví dụ $K$ tốt nhất có thể thay đổi hoặc điểm số tổng thể thấp hơn so với $K=5$ ở mục 3). Các tâm cụm lúc này sẽ có thêm thành phần tọa độ tuổi trung bình cho từng nhóm.

In [5]:
#Bài 3
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
Xs = StandardScaler().fit_transform(df[["thu_nhap", "diem_chi_tieu"]])

for init_type in ["random", "k-means++"]:
    dem = 0
    for rs in range(50):
        km = KMeans(n_clusters=5, init=init_type, n_init=1, random_state=rs).fit(Xs)
        if abs(km.inertia_ - 23.0) < 0.1:
            dem += 1
    print(f"Số lần đạt inertia 23.0 với init='{init_type}': {dem} / 50 lần")

Số lần đạt inertia 23.0 với init='random': 30 / 50 lần
Số lần đạt inertia 23.0 với init='k-means++': 50 / 50 lần


Phương pháp khởi tạo init="k-means++" đáng tin cậy hơn rất nhiều so với init="random" vì nó chủ động chọn các tâm ban đầu nằm cách xa nhau, giúp thuật toán hội tụ về nghiệm tối ưu toàn cục ổn định thay vì rơi vào các cực trị cục bộ kém chất lượng.

In [6]:
#Bài 4
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

os.makedirs("baitap08", exist_ok=True)

anh = load_sample_image("china.jpg")
diem = anh.reshape(-1, 3) / 255.0

rng = np.random.default_rng(0)
mau = diem[rng.choice(len(diem), 5000, replace=False)]

km = KMeans(n_clusters=16, n_init=4, random_state=0).fit(mau)
nhan = km.predict(diem)
anh_moi = km.cluster_centers_[nhan].reshape(anh.shape)

plt.imsave("baitap08/anh_goc.png", anh / 255.0)
plt.imsave("baitap08/anh_16_mau.png", anh_moi)

print("Đã lưu thành công ảnh gốc và ảnh 16 màu vào thư mục baitap08/")
print(f"Dung lượng tệp ảnh gốc: {os.path.getsize('baitap08/anh_goc.png') / 1024:.1f} KB")
print(f"Dung lượng tệp ảnh 16 màu: {os.path.getsize('baitap08/anh_16_mau.png') / 1024:.1f} KB")

Đã lưu thành công ảnh gốc và ảnh 16 màu vào thư mục baitap08/
Dung lượng tệp ảnh gốc: 537.3 KB
Dung lượng tệp ảnh 16 màu: 131.2 KB


Dung lượng: Giảm khoảng 6 lần so với ảnh gốc (mỗi điểm ảnh chỉ lưu 4 bit thay vì 24 bit). Chất lượng: Sai lệch màu rất thấp (trung bình hơn 7 mức sáng), mắt thường hầu như không nhận ra sự khác biệt so với ảnh gốc. Ý nghĩa: Minh họa khả năng cân bằng hiệu quả của K-Means giữa việc tiết kiệm dung lượng lưu trữ và duy trì độ trung thực của hình ảnh.

In [9]:
#Bài 5
import pandas as pd
from sklearn.cluster import DBSCAN

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]

for e in [0.2, 0.3, 0.4]:
    db = DBSCAN(eps=e, min_samples=5).fit(X)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_noise = list(db.labels_).count(-1)  
    print(f"eps = {e}: Số cụm tìm được = {n_clusters}, Số điểm nhiễu (-1) = {n_noise}")

eps = 0.2: Số cụm tìm được = 1, Số điểm nhiễu (-1) = 40
eps = 0.3: Số cụm tìm được = 1, Số điểm nhiễu (-1) = 10
eps = 0.4: Số cụm tìm được = 1, Số điểm nhiễu (-1) = 3


Với giá trị eps phù hợp (ví dụ xung quanh $0.2$ hoặc $0.3$), DBSCAN có khả năng tách chính xác hai vầng trăng cong tốt hơn rất nhiều so với K-Means (vì K-Means bị giới hạn bởi ranh giới tuyến tính phẳng). Vai trò của eps (bán kính vùng lân cận): Định nghĩa khoảng cách tối đa giữa hai điểm để được coi là nằm trong cùng một vùng lân cận. Nếu eps quá nhỏ, các điểm bị tách rời thành nhiều mảnh hoặc thành nhiễu; nếu eps quá lớn, hai vầng trăng sẽ bị dính liền vào nhau thành một cụm duy nhất.

In [10]:
#Bài 6
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X)
cum = km.labels_

ct = pd.crosstab(y, cum)
print("BẢNG CHÉO (CROSSTAB) GIỮA NHÃN THẬT VÀ CỤM K-MEANS")
print(ct)

BẢNG CHÉO (CROSSTAB) GIỮA NHÃN THẬT VÀ CỤM K-MEANS
col_0    0    1    2   3    4    5    6    7    8    9
row_0                                                 
0      176    0    0   0    0    0    2    0    0    0
1        0  100    0  54    1    2    0    1    0   24
2        1    8    3   2   13    0    0    0    2  148
3        0    7    7   0  154    0    0    2   13    0
4        0    2   11   3    0    0  165    0    0    0
5        0    0    0   0    2    1    2  136   41    0
6        1    3    0   0    0  177    0    0    0    0
7        0    2  174   2    0    0    0    1    0    0
8        0  100    5   6    2    2    0    4   52    3
9        0    1    8  20    6    0    0    6  139    0


Các chữ số có hình dạng đặc trưng và ít nét tương đồng với các số khác (như số 0 hoặc số 1) thường được K-Means gom khá gọn vào một cụm riêng biệt. Ngược lại, các chữ số có nét viết dễ nhầm lẫn hoặc cấu trúc gần giống nhau (ví dụ: 4 và 9, hoặc 3 và 8, hoặc 1 và 7) thường hay bị gom chung vào cùng một cụm do khoảng cách Euclidean giữa các điểm ảnh của chúng không quá lớn.